# 🛡️ PhishGuard — 20 Golden Features ML Training Pipeline (Google Colab)
### Defensive AI — MLOps Cloud Architecture

> **Catatan Rekayasa:** Seluruh proses pelatihan model (*training*) dijalankan di **Google Colab** dengan akselerasi multi-core vCPU/GPU gratis. Komputer lokal murni difungsikan sebagai **Inference Engine** yang ringan, aman, dan berlatensi mikro (<25 ms).

Notebook ini melatih model **RandomForestClassifier (150 Decision Trees)** dengan **20 Dimensi Fitur Emas Leksikal & Semantik Terpadu**:

| Kategori | Fitur yang Diekstraksi |
| :--- | :--- |
| **Struktural & Delimiter** | `url_length`, `hostname_length`, `path_length`, `count_dot`, `count_hyphen`, `count_at`, `count_question`, `count_equal`, `count_slash` |
| **Sintaksis & Reputasi** | `count_digits`, `count_subdomains`, `has_https`, `has_ip`, `has_port`, `has_url_shortener`, `is_suspicious_tld` |
| **Intelijen Leksikal & Teori Informasi** | `shannon_entropy` ⚡, `has_homograph` 🔤, `suspicious_words` 🚨, `indonesian_keywords` 🇮🇩 |

**Alur Pipeline:**
1. Setup dependensi & unduh otomatis dataset resmi PhiUSIIL (235k URL) dari UCI ML Repository
2. Ekstraksi 20 fitur paralel multi-core berkecepatan tinggi
3. Stratified Data Split (70% Train, 15% Validation, 15% Test)
4. Komparasi Ilmiah: **Logistic Regression (Baseline)** vs **Random Forest 150 Trees (Proposed)**
5. Evaluasi performa lengkap (Accuracy, Precision, Recall, F1, ROC-AUC) & visualisasi 300 DPI (`feature_importance.png`, `confusion_matrix.png`, `roc_curve.png`, `model_comparison.png`)
6. Analisis Kesalahan Klasifikasi (**False Positive & False Negative Error Analysis** + `error_analysis.png`)
7. Real-World Live Sanity Check (Uji coba URL nyata Indonesia & global)
8. Ekspor & unduh otomatis artefak `model_phishing.pkl`, `metrics.json`, dan arsip seluruh grafik (`phishguard_evaluation_plots.zip`).

In [ ]:
# [1] Setup Dependensi & Konfigurasi Lingkungan Google Colab
import os
import glob
import math
import json
import zipfile
import urllib.request
from collections import Counter
from urllib.parse import urlsplit

import joblib
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

print("✓ Lingkungan Google Colab Siap. Scikit-Learn versi:", sklearn.__version__)

In [ ]:
# [2] Akuisisi Dataset: Deteksi Lokal/Drive Colab atau Unduh Otomatis PhiUSIIL (UCI ML Repository)

dataset_df = None
dataset_source = "unknown"

# 1. Periksa apakah user sudah mengunggah CSV ke /content atau Google Drive
local_csvs = glob.glob("/content/**/*.csv", recursive=True)
phiusiil_local = [p for p in local_csvs if "phiusiil" in os.path.basename(p).lower()]
if phiusiil_local:
    print(f"✓ Menemukan dataset lokal/Drive di Colab: {phiusiil_local[0]}")
    dataset_df = pd.read_csv(phiusiil_local[0])
    dataset_source = "phiusiil"
else:
    # 2. Unduh otomatis PhiUSIIL Phishing URL Dataset (235.795 URL) resmi dari UCI Repository
    print("ℹ Mengunduh PhiUSIIL URL Dataset resmi dari UCI ML Repository...")
    UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip"
    ZIP_PATH = "phiusiil_dataset.zip"
    EXTRACT_DIR = "dataset_extracted"
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    try:
        if not glob.glob(f"{EXTRACT_DIR}/**/*.csv", recursive=True):
            req = urllib.request.Request(UCI_ZIP_URL, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as resp, open(ZIP_PATH, "wb") as out_f:
                out_f.write(resp.read())
            with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
                zip_ref.extractall(EXTRACT_DIR)
        extracted_csv = glob.glob(f"{EXTRACT_DIR}/**/*.csv", recursive=True)
        if extracted_csv:
            dataset_df = pd.read_csv(extracted_csv[0])
            dataset_source = "phiusiil"
            print(f"✓ Sukses memuat PhiUSIIL: {len(dataset_df):,} baris data!")
    except Exception as e:
        print(f"⚠️ Gagal mengunduh UCI zip ({e}), menggunakan dataset alternatif...")
        ALT_URL = "https://raw.githubusercontent.com/datasets/phishing-urls/main/data/dataset.csv"
        dataset_df = pd.read_csv(ALT_URL)
        dataset_source = "alternative"

# Preprocessing Awal: Drop baris kosong dan duplikasi URL
url_col = "URL" if "URL" in dataset_df.columns else dataset_df.columns[0]
label_col = "label" if "label" in dataset_df.columns else dataset_df.columns[1]

dataset_df.dropna(subset=[url_col, label_col], inplace=True)
sebelum_dedup = len(dataset_df)
dataset_df.drop_duplicates(subset=[url_col], inplace=True)
print(f"✓ Pembersihan data: {sebelum_dedup - len(dataset_df):,} URL duplikat dihapus.")
print(f"✓ Total baris URL unik siap diolah: {len(dataset_df):,}")

In [ ]:
# [3] Definisi 20 Fitur Emas Leksikal, Sintaksis, dan Semantik

SUSPICIOUS_KEYWORDS = [
    "login", "verify", "account", "secure", "update", "signin",
    "banking", "password", "wallet", "confirm", "recover", "suspended",
    "webscr", "ebayisapi", "paypal", "credential"
]

INDO_KEYWORDS = [
    "undian", "bansos", "prakerja", "dana", "dana-kaget", "pulsa",
    "kejutan", "hadiah", "bca", "bri", "bni", "mandiri", "gopay",
    "shopee", "linkaja", "kuota", "subsidi", "blt", "ojk", "kominfo",
    "klikbca", "brimo"
]

URL_SHORTENERS = set([
    "bit.ly", "tinyurl.com", "t.co", "goo.gl", "is.gd", "ow.ly",
    "buff.ly", "rebrand.ly", "cutt.ly", "shorturl.at", "tiny.cc",
    "lnkd.in", "s.id", "rb.gy", "v.gd", "clck.ru", "rotf.lol"
])

SUSPICIOUS_TLDS = set([
    "xyz", "top", "tk", "ml", "ga", "cf", "gq", "work", "click",
    "buzz", "fit", "rest", "cam", "kim", "bar", "icu", "club",
    "live", "surf", "monster", "cfd"
])

CONFUSABLES = set("\u0430\u0441\u0435\u043e\u0440\u0455\u0445\u0456\u0458\u0443")

def calculate_shannon_entropy(text: str) -> float:
    if not text:
        return 0.0
    length = len(text)
    counts = Counter(text)
    return round(-sum((c / length) * math.log2(c / length) for c in counts.values()), 3)

def extract_20_features(url_str: str) -> dict:
    raw = str(url_str).strip()
    lowered = raw.lower()
    try:
        parsed = urlsplit(raw if "://" in raw else f"//{raw}")
        host = (parsed.hostname or "").rstrip(".").lower()
        port = parsed.port
        path = parsed.path or ""
    except:
        host, port, path = "", None, ""

    # Deteksi raw IPv4 numerik
    parts = host.split(".")
    is_ip = 1 if len(parts) == 4 and all(p.isdigit() and 0 <= int(p) <= 255 for p in parts) else 0

    # Subdomain depth
    subdomain_count = max(0, len([p for p in parts if p]) - 2)

    # Keyword hits
    susp_hits = sum(lowered.count(kw) for kw in SUSPICIOUS_KEYWORDS)
    indo_hits = sum(lowered.count(kw) for kw in INDO_KEYWORDS)

    # Homograph & Punycode
    has_homograph = 1 if ("xn--" in host or any(c in CONFUSABLES for c in host)) else 0

    # TLD risk & Shortener
    tld = parts[-1] if parts else ""
    is_suspicious_tld = 1 if tld in SUSPICIOUS_TLDS else 0
    is_shortener = 1 if any(host == s or host.endswith("." + s) for s in URL_SHORTENERS) else 0

    # Port check
    has_port = 1 if port is not None else 0

    # Shannon Entropy
    entropy = calculate_shannon_entropy(host)

    return {
        "url_length": len(raw),
        "hostname_length": len(host),
        "path_length": len(path),
        "count_dot": raw.count("."),
        "count_hyphen": raw.count("-"),
        "count_at": raw.count("@"),
        "count_question": raw.count("?"),
        "count_equal": raw.count("="),
        "count_slash": raw.count("/"),
        "count_digits": sum(char.isdigit() for char in raw),
        "count_subdomains": subdomain_count,
        "has_https": 1 if lowered.startswith("https://") else 0,
        "has_ip": is_ip,
        "has_port": has_port,
        "shannon_entropy": entropy,
        "has_homograph": has_homograph,
        "is_suspicious_tld": is_suspicious_tld,
        "has_url_shortener": is_shortener,
        "suspicious_words": susp_hits,
        "indonesian_keywords": indo_hits
    }

FEATURE_COLS = [
    "url_length", "hostname_length", "path_length", "count_dot", "count_hyphen",
    "count_at", "count_question", "count_equal", "count_slash", "count_digits",
    "count_subdomains", "has_https", "has_ip", "has_port", "shannon_entropy",
    "has_homograph", "is_suspicious_tld", "has_url_shortener", "suspicious_words",
    "indonesian_keywords"
]

print(f"✓ Konfigurasi 20 Fitur Emas Berhasil Diinisialisasi!")
for i, f in enumerate(FEATURE_COLS, 1):
    print(f"  {i:02d}. {f}")

In [ ]:
# [4] Ekstraksi Fitur Paralel Berkecepatan Tinggi (Colab Multi-Core CPU)
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

urls = dataset_df[url_col].astype(str).tolist()
raw_labels = dataset_df[label_col].tolist()

# Normalisasi Label: 1 = Phishing, 0 = Legitimate
# Pada dataset resmi PhiUSIIL UCI: Label 0 = Phishing, 1 = Legitimate
if dataset_source == "phiusiil" or len(dataset_df) > 200000:
    print("ℹ Deteksi dataset PhiUSIIL UCI: memetakan Label 0 -> 1 (Phishing), Label 1 -> 0 (Legitimate).")
    y = np.array([1 if int(l) == 0 else 0 for l in raw_labels])
else:
    y = np.array([1 if int(l) == 1 else 0 for l in raw_labels])

print(f"Mengekstrak 20 fitur dari {len(urls):,} URL dengan {os.cpu_count() or 2} thread CPU...")
with ThreadPoolExecutor(max_workers=os.cpu_count() or 2) as executor:
    extracted_list = list(tqdm(executor.map(extract_20_features, urls), total=len(urls)))

X = pd.DataFrame(extracted_list)[FEATURE_COLS]

# Stratified Data Splitting: 70% Train, 15% Validation, 15% Held-Out Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("=" * 50)
print("✓ Ekstraksi dan Pembagian Data Selesai!")
print(f"  - Train Set      : {len(X_train):,} baris ({np.mean(y_train)*100:.1f}% Phishing)")
print(f"  - Validation Set : {len(X_val):,} baris ({np.mean(y_val)*100:.1f}% Phishing)")
print(f"  - Test Set       : {len(X_test):,} baris ({np.mean(y_test)*100:.1f}% Phishing)")
print("=" * 50)

In [ ]:
# [5] Pelatihan Model: Logistic Regression (Baseline) vs RandomForestClassifier (150 Trees)

# 1. Model Baseline: Logistic Regression
print("[1/2] Melatih Model Baseline: Logistic Regression...")
baseline_model = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)
baseline_f1 = f1_score(y_test, baseline_pred, zero_division=0)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"      Baseline Accuracy: {baseline_acc*100:.2f}% | F1-Score: {baseline_f1*100:.2f}%")

# 2. Model Utama yang Dipilih: RandomForestClassifier (150 Decision Trees)
print("[2/2] Melatih Model Utama: RandomForestClassifier (150 Trees Ensemble)...")
rf_model = RandomForestClassifier(
    n_estimators=150,
    criterion="gini",
    max_features="sqrt",
    min_samples_split=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_model.fit(X_train, y_train)
print("✓ Pelatihan RandomForestClassifier 150 Trees Selesai!")

In [ ]:
# [6] Evaluasi Ilmiah, Komparasi Model & Visualisasi Grafik pada Data Uji (Test Set)

# Hitung metrik lengkap untuk Baseline (Logistic Regression)
baseline_prob = baseline_model.predict_proba(X_test)[:, 1]
baseline_prec = precision_score(y_test, baseline_pred, zero_division=0)
baseline_rec = recall_score(y_test, baseline_pred, zero_division=0)
baseline_roc = roc_auc_score(y_test, baseline_prob)

# Hitung metrik lengkap untuk Proposed Model (Random Forest 150 Trees)
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()
fpr_val = fp / (fp + tn) if (fp + tn) > 0 else 0.0
fnr_val = fn / (fn + tp) if (fn + tp) > 0 else 0.0

print("=" * 75)
print("          TABEL HASIL KOMPARASI METODE (HELD-OUT TEST SET)")
print("=" * 75)
print(f"{'Model':<32} | {'Acc':>7} | {'Prec':>7} | {'Rec':>7} | {'F1':>7} | {'AUC':>7}")
print("-" * 75)
print(f"{'Logistic Regression (Baseline)':<32} | {baseline_acc*100:6.2f}% | {baseline_prec*100:6.2f}% | {baseline_rec*100:6.2f}% | {baseline_f1*100:6.2f}% | {baseline_roc:7.4f}")
print(f"{'Random Forest (PhishGuard)':<32} | {acc*100:6.2f}% | {prec*100:6.2f}% | {rec*100:6.2f}% | {f1*100:6.2f}% | {roc:7.4f}")
print("-" * 75)
print(f"Peningkatan Performa Random Forest -> Akurasi: +{(acc - baseline_acc)*100:.2f}% | F1-Score: +{(f1 - baseline_f1)*100:.2f}%")
print(f"Confusion Matrix -> TN: {tn:,} | FP: {fp:,} (FPR: {fpr_val*100:.2f}%) | FN: {fn:,} (FNR: {fnr_val*100:.2f}%) | TP: {tp:,}")
print("=" * 75)
print("\nClassification Report (Random Forest 150 Trees):")
print(classification_report(y_test, y_pred, target_names=["Legitimate (0)", "Phishing (1)"], digits=4))

# Visualisasi 1: 20 Feature Importance Ranking
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
plt.figure(figsize=(10, 7))
importances.plot(kind="barh", color="#1D4ED8")
plt.title("PhishGuard 20 Golden Feature Importances (Random Forest 150 Trees)", fontsize=13, fontweight="bold")
plt.xlabel("Gini Impurity Reduction Contribution")
plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=300)
plt.show()

# Visualisasi 2: Confusion Matrix Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Legitimate (0)", "Phishing (1)"],
    yticklabels=["Legitimate (0)", "Phishing (1)"]
)
plt.title(f"Confusion Matrix Test Set ({len(y_test):,} Sampel)", fontweight="bold")
plt.xlabel("Prediksi Model")
plt.ylabel("Label Sebenarnya")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

# Visualisasi 3: Kurva ROC-AUC (Random Forest vs Logistic Regression Baseline)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob)
fpr_lr, tpr_lr, _ = roc_curve(y_test, baseline_prob)

plt.figure(figsize=(7, 5.5))
plt.plot(fpr_rf, tpr_rf, color="#1D4ED8", lw=2.5, label=f"Random Forest PhishGuard (AUC = {roc:.4f})")
plt.plot(fpr_lr, tpr_lr, color="#F59E0B", lw=2.0, linestyle="-.", label=f"Logistic Regression Baseline (AUC = {baseline_roc:.4f})")
plt.plot([0, 1], [0, 1], color="#9CA3AF", lw=1.5, linestyle="--", label="Random Classifier (AUC = 0.5000)")
plt.scatter([fpr_val], [rec], color="#DC2626", s=90, zorder=5, label=f"Threshold 0.50 (FPR={fpr_val*100:.2f}%, TPR={rec*100:.2f}%)")
plt.annotate(
    f"Ambang Batas 0.50\n(FPR={fpr_val*100:.2f}%, TPR={rec*100:.2f}%)",
    xy=(fpr_val, rec),
    xytext=(min(0.45, fpr_val + 0.12), max(0.55, rec - 0.15)),
    arrowprops=dict(arrowstyle="->", color="#DC2626", lw=1.5),
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#FEF2F2", edgecolor="#DC2626", lw=1.2),
    fontsize=9,
    fontweight="bold",
    color="#991B1B"
)
plt.title("Receiver Operating Characteristic (ROC) Curve — Held-Out Test Set", fontsize=12, fontweight="bold")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate / Recall (TPR)")
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.04])
plt.grid(linestyle="--", alpha=0.4)
plt.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=300)
plt.show()

# Visualisasi 4: Grafik Perbandingan Metrik Model (Baseline vs Random Forest)
metrics_names = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
lr_scores = [baseline_acc * 100, baseline_prec * 100, baseline_rec * 100, baseline_f1 * 100, baseline_roc * 100]
rf_scores = [acc * 100, prec * 100, rec * 100, f1 * 100, roc * 100]

x_pos = np.arange(len(metrics_names))
bar_width = 0.36

plt.figure(figsize=(9, 5))
bars_lr = plt.bar(x_pos - bar_width / 2, lr_scores, bar_width, label="Logistic Regression (Baseline)", color="#94A3B8")
bars_rf = plt.bar(x_pos + bar_width / 2, rf_scores, bar_width, label="Random Forest 150 Trees (PhishGuard)", color="#1D4ED8")

for bar in bars_lr:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 0.4, f"{h:.2f}%", ha="center", va="bottom", fontsize=8.5, color="#334155")
for bar in bars_rf:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 0.4, f"{h:.2f}%", ha="center", va="bottom", fontsize=8.5, fontweight="bold", color="#1E3A8A")

plt.xticks(x_pos, metrics_names, fontsize=10, fontweight="bold")
plt.ylabel("Skor Evaluasi (%)")
plt.ylim(75, 105)
plt.title("Perbandingan Kinerja Model: Logistic Regression vs Random Forest", fontsize=12, fontweight="bold")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=300)
plt.show()

In [ ]:
# [7] Analisis Mendalam False Positive (FP) & False Negative (FN)

eval_df = X_test.copy()
eval_df["url"] = [urls[idx] for idx in X_test.index]
eval_df["actual"] = y_test
eval_df["predicted"] = y_pred
eval_df["phishing_prob"] = np.round(y_prob, 4)

tn_df = eval_df[(eval_df["actual"] == 0) & (eval_df["predicted"] == 0)]
fp_df = eval_df[(eval_df["actual"] == 0) & (eval_df["predicted"] == 1)].sort_values("phishing_prob", ascending=False)
fn_df = eval_df[(eval_df["actual"] == 1) & (eval_df["predicted"] == 0)].sort_values("phishing_prob", ascending=True)
tp_df = eval_df[(eval_df["actual"] == 1) & (eval_df["predicted"] == 1)]

print("=" * 75)
print("           ANALISIS KESALAHAN KLASIFIKASI (ERROR ANALYSIS)")
print("=" * 75)
print(f"Total Sampel Uji   : {len(eval_df):,}")
print(f"False Positive (FP): {len(fp_df):,} URL aman yang salah ditebak Phishing (FPR = {fpr_val*100:.2f}%)")
print(f"False Negative (FN): {len(fn_df):,} URL phishing yang lolos ditebak Aman (FNR = {fnr_val*100:.2f}%)")

# Bandingkan karakteristik rata-rata fitur antar kelompok
profile_cols = ["url_length", "hostname_length", "path_length", "shannon_entropy", "count_slash", "has_https", "suspicious_words"]
summary_profiles = pd.DataFrame({
    "TN (Aman Benar)": tn_df[profile_cols].mean(),
    "FP (Aman -> Phishing)": fp_df[profile_cols].mean() if len(fp_df) > 0 else pd.Series(0, index=profile_cols),
    "FN (Phishing -> Aman)": fn_df[profile_cols].mean() if len(fn_df) > 0 else pd.Series(0, index=profile_cols),
    "TP (Phishing Benar)": tp_df[profile_cols].mean(),
}).round(3)

print("\nPerbandingan Rata-Rata Fitur Utama per Kategori Prediksi:")
print(summary_profiles.to_string())

print("\n--- Contoh 10 Sampel FALSE POSITIVE (URL Aman yang Tertandai Phishing) ---")
if len(fp_df) == 0:
    print("Tidak ditemukan kasus False Positive pada data uji.")
else:
    for i, (_, row) in enumerate(fp_df.head(10).iterrows(), 1):
        short_url = row["url"][:95] + ("..." if len(row["url"]) > 95 else "")
        print(f"{i:2d}. [Prob={row['phishing_prob']:.4f} | len={int(row['url_length'])}, path={int(row['path_length'])}, https={int(row['has_https'])}, ent={row['shannon_entropy']:.2f}] {short_url}")

print("\n--- Contoh 10 Sampel FALSE NEGATIVE (URL Phishing yang Lolos sebagai Aman) ---")
if len(fn_df) == 0:
    print("Tidak ditemukan kasus False Negative pada data uji.")
else:
    for i, (_, row) in enumerate(fn_df.head(10).iterrows(), 1):
        short_url = row["url"][:95] + ("..." if len(row["url"]) > 95 else "")
        print(f"{i:2d}. [Prob={row['phishing_prob']:.4f} | len={int(row['url_length'])}, path={int(row['path_length'])}, https={int(row['has_https'])}, ent={row['shannon_entropy']:.2f}] {short_url}")

# Visualisasi 5: Grafik Error Analysis (Perbandingan Karakteristik & Distribusi Probabilitas Kesalahan)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# Panel Kiri: Rata-rata Panjang URL & Path pada 4 kelompok prediksi
groups = ["TN\n(Aman Benar)", "FP\n(Salah Blokir)", "FN\n(Lolos)", "TP\n(Phishing Benar)"]
mean_url_lens = [
    tn_df["url_length"].mean(),
    fp_df["url_length"].mean() if len(fp_df) > 0 else 0,
    fn_df["url_length"].mean() if len(fn_df) > 0 else 0,
    tp_df["url_length"].mean()
]
mean_path_lens = [
    tn_df["path_length"].mean(),
    fp_df["path_length"].mean() if len(fp_df) > 0 else 0,
    fn_df["path_length"].mean() if len(fn_df) > 0 else 0,
    tp_df["path_length"].mean()
]

gx = np.arange(len(groups))
axes[0].bar(gx - 0.18, mean_url_lens, 0.36, label="Rata-rata url_length", color="#2563EB")
axes[0].bar(gx + 0.18, mean_path_lens, 0.36, label="Rata-rata path_length", color="#60A5FA")
axes[0].set_xticks(gx)
axes[0].set_xticklabels(groups, fontsize=9)
axes[0].set_ylabel("Jumlah Karakter")
axes[0].set_title("Profil Panjang URL & Path per Kuadran Prediksi", fontweight="bold")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)
axes[0].legend()

# Panel Kanan: Distribusi Skor Probabilitas Kasus Salah (FP vs FN)
if len(fp_df) > 0:
    axes[1].hist(fp_df["phishing_prob"], bins=15, alpha=0.7, color="#F59E0B", label=f"False Positive (n={len(fp_df)})")
if len(fn_df) > 0:
    axes[1].hist(fn_df["phishing_prob"], bins=15, alpha=0.7, color="#DC2626", label=f"False Negative (n={len(fn_df)})")
axes[1].axvline(0.50, color="#111827", linestyle="--", lw=1.8, label="Threshold Keputusan (0.50)")
axes[1].set_xlim(0.0, 1.0)
axes[1].set_xlabel("Probabilitas Prediksi Phishing Model")
axes[1].set_ylabel("Jumlah Sampel")
axes[1].set_title("Distribusi Keyakinan Model pada Kasus FP & FN", fontweight="bold")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)
axes[1].legend()

plt.tight_layout()
plt.savefig("error_analysis.png", dpi=300)
plt.show()

In [ ]:
# [8] Real-World Live Sanity Check: Uji Coba Langsung 5 URL Nyata

test_samples = [
    ("https://google.com/search?q=cybersecurity", "Google Search"),
    ("https://github.com/security", "GitHub Security"),
    ("http://192.168.1.100/login-update-paypal-banking-verify", "IP Host Attack"),
    ("http://dana-kaget-bansos-bri.xyz/login", "Indonesian Phishing Token"),
    ("http://xn--pple-43d.com/signin", "Homograph Cyrillic Spoof")
]

print("=" * 65)
print("        PENGUJIAN LANGSUNG URL NYATA PADA MODEL BARU")
print("=" * 65)
for sample_url, desc in test_samples:
    feats = extract_20_features(sample_url)
    feat_vector = pd.DataFrame([feats])[FEATURE_COLS]
    score = rf_model.predict_proba(feat_vector)[0][1]
    verdict = "PHISHING ⚠️" if score >= 0.50 else "LEGITIMATE 🛡️"
    print(f"[{desc}]")
    print(f"  URL   : {sample_url}")
    print(f"  Hasil : {verdict} (Skor Keyakinan: {score*100:.1f}%)")
    print("-" * 65)

In [ ]:
# [9] Ekspor & Unduh Artefak Model, Metrik, dan Grafik (Google Colab)

# 1. Simpan Bundle model_phishing.pkl
output_bundle = {
    "model": rf_model,
    "feature_names": FEATURE_COLS,
    "threshold": 0.50,
    "algorithm": "RandomForestClassifier (150 Trees)",
    "metadata": {
        "dataset": "PhiUSIIL Phishing URL Dataset (UCI ML Repository)",
        "features_count": len(FEATURE_COLS),
        "n_estimators": 150,
        "training_environment": "Google Colab"
    }
}

joblib.dump(output_bundle, "model_phishing.pkl")
print(f"✓ Sukses mengekspor model_phishing.pkl ({os.path.getsize('model_phishing.pkl') / 1e6:.1f} MB)")

# 2. Simpan metrics.json
feature_rankings = [
    {"feature": feat, "importance": round(float(imp), 4), "rank": i + 1}
    for i, (feat, imp) in enumerate(importances.sort_values(ascending=False).items())
]

metrics_dict = {
    "dataset": {
        "name": "PhiUSIIL Phishing URL Dataset (UCI ML Repository)",
        "total_samples": len(dataset_df),
        "features_count": len(FEATURE_COLS),
        "split_strategy": "Stratified Split (70/15/15)"
    },
    "model": {
        "algorithm": "RandomForestClassifier",
        "n_estimators": 150,
        "criterion": "gini",
        "threshold": 0.50
    },
    "test_metrics": {
        "accuracy": round(float(acc), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "f1_score": round(float(f1), 4),
        "roc_auc": round(float(roc), 4)
    },
    "baseline_metrics": {
        "algorithm": "LogisticRegression",
        "accuracy": round(float(baseline_acc), 4),
        "precision": round(float(baseline_prec), 4),
        "recall": round(float(baseline_rec), 4),
        "f1_score": round(float(baseline_f1), 4),
        "roc_auc": round(float(baseline_roc), 4)
    },
    "confusion_matrix": {
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "false_positive_rate": round(float(fpr_val), 6),
        "false_negative_rate": round(float(fnr_val), 6)
    },
    "feature_importance": feature_rankings
}

with open("metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_dict, f, indent=2)
print("✓ Sukses mengekspor metrics.json")

# 3. Kemas seluruh grafik evaluasi ke arsip ZIP agar mudah diunduh di Colab
plot_files = [
    "feature_importance.png",
    "confusion_matrix.png",
    "roc_curve.png",
    "model_comparison.png",
    "error_analysis.png",
]
with zipfile.ZipFile("phishguard_evaluation_plots.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for pf in plot_files:
        if os.path.isfile(pf):
            zf.write(pf)
print("✓ Sukses mengemas 5 grafik evaluasi ke phishguard_evaluation_plots.zip")

print("=" * 65)
print("🎉 TRAINING & EVALUASI GOOGLE COLAB 20-FITUR SELESAI!")
print("File output yang dihasilkan:")
print("  1. model_phishing.pkl")
print("  2. metrics.json")
print("  3. phishguard_evaluation_plots.zip (berisi 5 grafik PNG 300 DPI)")
print("=" * 65)

# 4. Unduh otomatis jika dijalankan di lingkungan Google Colab
try:
    from google.colab import files
    print("Mengunduh artefak dari Google Colab...")
    files.download("model_phishing.pkl")
    files.download("metrics.json")
    files.download("phishguard_evaluation_plots.zip")
except ImportError:
    print("ℹ Di luar Google Colab: semua file tersedia di direktori kerja saat ini.")